In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework") \
    .getOrCreate()

26/03/04 05:22:05 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### Spark version

In [4]:
spark.version

'3.5.5'

In [5]:
!ls -ltrh /home/iceberg/warehouse/source/yellow_tripdata_2025-11.parquet

-rw-r--r-- 1 root root 68M Dec 19 15:51 /home/iceberg/warehouse/source/yellow_tripdata_2025-11.parquet


In [6]:
df = spark.read.parquet("/home/iceberg/warehouse/source/yellow_tripdata_2025-11.parquet")

In [7]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [8]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

### Avg. size of parquet

In [10]:
df.repartition(4).write.parquet("/home/iceberg/warehouse/staging/2025/11/", mode="overwrite")

In [11]:
!ls -ltrh /home/iceberg/warehouse/staging/2025/11/

total 101M
-rw-r--r-- 1 root root 25M Mar  4 05:23 part-00003-68917d54-28c7-4f0d-9e91-06f5b5c46cf7-c000.snappy.parquet
-rw-r--r-- 1 root root 25M Mar  4 05:23 part-00002-68917d54-28c7-4f0d-9e91-06f5b5c46cf7-c000.snappy.parquet
-rw-r--r-- 1 root root 25M Mar  4 05:23 part-00000-68917d54-28c7-4f0d-9e91-06f5b5c46cf7-c000.snappy.parquet
-rw-r--r-- 1 root root 25M Mar  4 05:23 part-00001-68917d54-28c7-4f0d-9e91-06f5b5c46cf7-c000.snappy.parquet
-rw-r--r-- 1 root root   0 Mar  4 05:23 _SUCCESS


In [12]:
df = spark.read.parquet("/home/iceberg/warehouse/staging/2025/11/")

### Number of trips on 15th Nov

In [13]:
df.withColumn("pickup_date", F.to_date(F.col("tpep_pickup_datetime"))) \
    .filter(F.col("pickup_date") == "2025-11-15") \
    .count()

162604

### Longest trip

In [14]:
df.createOrReplaceTempView("yellow_trips")

In [15]:
%%sql

select 
    max((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) as max_duration
from yellow_trips


26/03/04 05:23:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


max_duration
90.64666666666666


### Least frequent pickup location zone

In [16]:
df_zone = spark.read \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .csv("/home/iceberg/warehouse/source/taxi_zone_lookup.csv")

In [17]:
df_zone.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [25]:
df_joined = df.join(df_zone, F.col("PULocationID") == F.col("LocationID"), how="inner")

In [32]:
df_joined.groupBy("Zone") \
    .agg(F.count("PULocationID").alias("pickup_frequency")) \
    .orderBy(F.asc("pickup_frequency")) \
    .limit(10) \
    .show()

+--------------------+----------------+
|                Zone|pickup_frequency|
+--------------------+----------------+
|Eltingville/Annad...|               1|
|       Arden Heights|               1|
|Governor's Island...|               1|
|       Port Richmond|               3|
| Green-Wood Cemetery|               4|
|       Rikers Island|               4|
|         Great Kills|               4|
|   Rossville/Woodrow|               4|
|         Jamaica Bay|               5|
|         Westerleigh|              12|
+--------------------+----------------+

